##### Test LLama3.2 model

In [ ]:
import os
from pathlib import Path

import requests
import torch
from PIL import Image
from transformers import MllamaForConditionalGeneration, AutoProcessor

ROOT = Path.cwd()
if not (ROOT / "visionguard").exists():
    ROOT = ROOT.parent

token_file_path = ROOT / "data" / "hf_token.txt"
cache_directory = ROOT / "cache" / "huggingface"

huggingface_token = os.environ.get("HF_TOKEN")
if huggingface_token is None and token_file_path.exists():
    huggingface_token = token_file_path.read_text().strip()

if huggingface_token is None:
    raise ValueError("Set HF_TOKEN env var or create data/hf_token.txt")

model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device="cuda",
    token=huggingface_token,
    cache_dir=str(cache_directory),
)
processor = AutoProcessor.from_pretrained(
    model_id,
    token=huggingface_token,
    cache_dir=str(cache_directory),
)

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/0052a70beed5bf71b92610a43a52df6d286cd5f3/diffusers/rabbit.jpg"
image = Image.open(requests.get(url, stream=True).raw)

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Can you please describe this image in just one sentence?"}
    ]}
]

input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to(model.device)
output = model.generate(**inputs, max_new_tokens=70)

print(processor.decode(output[0][inputs["input_ids"].shape[-1]:]))
